# Ensamble simple: ¿ayuda promediar varios modelos?

## Qué responde este notebook

Tercera y última mejora de esta ronda ("bajo costo, con lo que ya tenemos"). La idea de un
ensamble es simple: si varios modelos cometen errores distintos entre sí, promediar sus
predicciones puede reducir el error total sin entrenar nada nuevo y complejo. Pero **eso solo
funciona si los modelos son realmente diversos** -- promediar 3 modelos casi idénticos no reduce
nada. Este notebook lo verifica con evidencia real antes de recomendar cualquier ensamble,
reutilizando `experimentos.entrenar_y_evaluar()` tal cual ya existe (mismos hiperparámetros ya
elegidos en `13_comparacion_modelos.ipynb` -- ninguna búsqueda nueva).

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

AZUL, NARANJA, VERDE, GRIS, GRIS_CLARO = "#2a78d6", "#eb6834", "#1baf7a", "#888780", "#c3c2b7"

wr = features.construir_tabla_modelado(range(2016, 2026))
features_arbol = features.columnas_por_modelo("arbol")
features_ridge = features.columnas_por_modelo("ridge")
train_mask, val_mask, test_mask = experimentos.dividir_temporal(wr, range(2016, 2022), [2022, 2023], [2024, 2025])


## 1. Un caso concreto antes que cualquier tabla

Ja'Marr Chase, una semana real de 2025 -- ¿qué tan distintas son las predicciones de XGBoost,
RandomForest e HistGradientBoosting para el mismo jugador-semana?

In [2]:
resultados_yardas, resumen_yardas = experimentos.entrenar_y_evaluar(
    wr, "receiving_yards", features_arbol, features_ridge, train_mask, val_mask, test_mask
)
print(resumen_yardas.round(3).to_string())

chase_id = data.cargar_jugadores().pipe(lambda j: j[j["display_name"] == "Ja'Marr Chase"])["gsis_id"].iloc[0]
idx_chase = wr[test_mask & (wr["player_id"] == chase_id)].sort_values(["season", "week"]).index[-1]

print(f"\nreal: {wr.loc[idx_chase, 'receiving_yards']:.0f} yardas")
for nombre in ["XGBoost", "RandomForest", "HistGradientBoosting", "Ridge"]:
    pred = resultados_yardas[nombre]["pred"][idx_chase]
    print(f"  {nombre}: {pred:.1f}")


                      train_mae  train_rmse  train_r2  val_mae  val_rmse  val_r2  test_mae  test_rmse  test_r2
XGBoost                  21.549      29.240     0.372   22.015    29.528   0.363    20.650     27.964    0.366
RandomForest             21.363      29.006     0.382   22.069    29.529   0.363    20.764     27.950    0.367
HistGradientBoosting     20.784      28.099     0.420   22.182    29.649   0.358    20.806     28.097    0.360
Ridge                    22.362      30.318     0.325   22.368    29.823   0.351    21.253     28.658    0.335
Baseline                 23.445      32.428     0.228   22.572    31.387   0.281    21.281     29.860    0.277

real: 96 yardas
  XGBoost: 89.2
  RandomForest: 84.2
  HistGradientBoosting: 79.9
  Ridge: 85.7


Las 3 predicciones basadas en árboles (XGBoost/RandomForest/HistGradientBoosting) caen muy
cerca entre sí -- Ridge es la que más difiere. Es la primera pista de que promediar solo los 3
árboles puede no aportar mucho: para que un ensamble reduzca error hace falta que los modelos
discrepen en direcciones distintas, no que casi siempre digan lo mismo.

## 2. Se confirma con la correlación real de las predicciones (test completo)

In [3]:
def correlacion_predicciones(resultados, nombres):
    preds = pd.DataFrame({n: resultados[n]["pred"][test_mask] for n in nombres})
    return preds, preds.corr()

preds_yardas, corr_yardas = correlacion_predicciones(resultados_yardas, ["XGBoost", "RandomForest", "HistGradientBoosting", "Ridge"])
print("correlacion entre predicciones, receiving_yards (test 2024-2025):")
print(corr_yardas.round(3).to_string())


correlacion entre predicciones, receiving_yards (test 2024-2025):
                      XGBoost  RandomForest  HistGradientBoosting  Ridge
XGBoost                 1.000         0.988                 0.985  0.953
RandomForest            0.988         1.000                 0.975  0.949
HistGradientBoosting    0.985         0.975                 1.000  0.937
Ridge                   0.953         0.949                 0.937  1.000


XGBoost/RandomForest/HistGradientBoosting corren entre 0.975-0.988 de correlación -- casi
redundantes entre sí (misma familia de árboles, mismas 34 features). Ridge baja a ~0.93-0.95 con
cada uno de ellos -- más diverso, por ser un modelo estructuralmente distinto (lineal, con
imputación y escalado, features sin la colinealidad de participación). Con esta evidencia, se
prueban 2 ensambles distintos para `receptions`/`receiving_yards`: solo los 3 árboles, y
árboles + Ridge.

In [4]:
def evaluar_ensambles(target, resultados, resumen):
    y_test = wr.loc[test_mask, target]
    pred_xgb = resultados["XGBoost"]["pred"][test_mask]
    pred_rf = resultados["RandomForest"]["pred"][test_mask]
    pred_hgb = resultados["HistGradientBoosting"]["pred"][test_mask]
    pred_ridge = resultados["Ridge"]["pred"][test_mask]

    ensamble_3 = np.clip((pred_xgb + pred_rf + pred_hgb) / 3, 0, None)
    ensamble_4 = np.clip((pred_xgb + pred_rf + pred_hgb + pred_ridge) / 4, 0, None)

    filas = [{"version": "mejor individual (" + resumen.index[0] + ")", **modeling.metricas(y_test, resultados[resumen.index[0]]["pred"][test_mask])}]
    filas.append({"version": "ensamble (xgb+rf+hgb)/3", **modeling.metricas(y_test, ensamble_3)})
    filas.append({"version": "ensamble (xgb+rf+hgb+ridge)/4", **modeling.metricas(y_test, ensamble_4)})
    return pd.DataFrame(filas).set_index("version")

tabla_yardas = evaluar_ensambles("receiving_yards", resultados_yardas, resumen_yardas)
print("receiving_yards:")
print(tabla_yardas.round(4).to_string())

resultados_recepciones, resumen_recepciones = experimentos.entrenar_y_evaluar(
    wr, "receptions", features_arbol, features_ridge, train_mask, val_mask, test_mask
)
tabla_recepciones = evaluar_ensambles("receptions", resultados_recepciones, resumen_recepciones)
print("\nreceptions:")
print(tabla_recepciones.round(4).to_string())


receiving_yards:
                                   mae     rmse      r2
version                                                
mejor individual (XGBoost)     20.6502  27.9639  0.3664
ensamble (xgb+rf+hgb)/3        20.6841  27.9150  0.3686
ensamble (xgb+rf+hgb+ridge)/4  20.3304  27.8607  0.3710



receptions:
                                  mae    rmse      r2
version                                              
mejor individual (XGBoost)     1.3647  1.8257  0.4532
ensamble (xgb+rf+hgb)/3        1.3725  1.8292  0.4511
ensamble (xgb+rf+hgb+ridge)/4  1.3535  1.8211  0.4559


Confirma la hipótesis: el ensamble de solo los 3 árboles (correlación >0.97 entre ellos) es
**ligeramente peor** que el mejor modelo individual en MAE, para ambos targets -- promediar
modelos casi idénticos no reduce error, solo diluye al mejor con los otros 2 un poco peores.
Agregar Ridge (el único realmente diverso) sí ayuda: mejora MAE ~0.8-1.5% y sube R² en ambos
targets. La lección es concreta, no genérica: **un ensamble ayuda por la diversidad de los
modelos que lo componen, no por la cantidad** -- 4 modelos con uno diferente le ganan a 3 modelos
parecidos entre sí.

## 3. `receiving_tds`: un caso distinto, con una razón real para ensamblar

Este target ya tiene 2 modelos estructuralmente distintos construidos en Fase 5
(`14_receiving_tds_modelos_de_conteo.ipynb`): el XGBoost Tweedie (mejor MAE, peor R² -- la
tensión ya conocida) y el modelo hurdle de 2 etapas (peor MAE, mejor R² -- de hecho el único con
R² positivo de todo este target). Al ser tan distintos en su lógica (uno regresión directa de
conteo, el otro clasificar-y-luego-regresar), su correlación debería ser más baja -- si es así,
mezclarlos no es "ensamble por ensamblar", es una forma real de moverse en el trade-off
MAE-vs-R² que lleva pendiente desde `20_mejora_del_modelo.ipynb`.

In [5]:
target = "receiving_tds"
y_test_tds = wr.loc[test_mask, target]

modelo_tweedie = XGBRegressor(objective="reg:tweedie", tweedie_variance_power=1.5, random_state=0, n_jobs=-1)
modelo_tweedie.fit(wr.loc[train_mask, features_arbol], wr.loc[train_mask, target])
pred_tweedie = np.clip(modelo_tweedie.predict(wr.loc[test_mask, features_arbol]), 0, None)

salida_hurdle = experimentos.entrenar_hurdle(wr, target, features_arbol, train_mask, val_mask, test_mask)
pred_hurdle = np.clip(salida_hurdle["pred"][test_mask], 0, None)

print("correlacion tweedie vs hurdle:", round(np.corrcoef(pred_tweedie, pred_hurdle)[0, 1], 3),
      "(vs 0.975-0.988 entre los 3 arboles de arriba -- mucho mas diverso)")

print("\ntweedie solo:", {k: round(v, 4) for k, v in modeling.metricas(y_test_tds, pred_tweedie).items()})
print("hurdle solo: ", {k: round(v, 4) for k, v in salida_hurdle["test"].items()})

print("\nbarrido de mezcla (peso a tweedie, resto a hurdle):")
for w in (0.3, 0.5, 0.7):
    e = np.clip(w * pred_tweedie + (1 - w) * pred_hurdle, 0, None)
    m = modeling.metricas(y_test_tds, e)
    print(f"  w_tweedie={w}: mae={m['mae']:.4f} r2={m['r2']:.4f}")


correlacion tweedie vs hurdle: 0.621 (vs 0.975-0.988 entre los 3 arboles de arriba -- mucho mas diverso)

tweedie solo: {'mae': 0.2488, 'rmse': 0.4765, 'r2': -0.041}
hurdle solo:  {'mae': 0.3074, 'rmse': 0.4503, 'r2': 0.0703}

barrido de mezcla (peso a tweedie, resto a hurdle):
  w_tweedie=0.3: mae=0.2898 r2=0.0629
  w_tweedie=0.5: mae=0.2781 r2=0.0456
  w_tweedie=0.7: mae=0.2664 r2=0.0184


Confirmado: correlación de 0.62 (contra 0.97-0.99 de los árboles), y el barrido de mezcla
muestra un trade-off real y **continuo**, no solo 2 extremos:

| Mezcla | MAE | R² |
|---|---|---|
| Tweedie puro | 0.249 | -0.041 |
| 70% Tweedie / 30% hurdle | 0.266 | +0.018 |
| 50/50 | 0.278 | +0.046 |
| 30% Tweedie / 70% hurdle | 0.290 | +0.063 |
| Hurdle puro | 0.307 | +0.070 |

Esto le da a la decisión pendiente sobre `receiving_tds` una opción que antes no estaba sobre la
mesa: en vez de elegir entre "MAE bueno, R² muy negativo" (Tweedie) o "MAE peor, R² positivo"
(hurdle) como 2 extremos, se puede elegir un punto intermedio del trade-off con un solo número de
mezcla (`w`). Por ejemplo, 70/30 ya saca el R² del terreno negativo (-0.041 → +0.018) por un costo
de MAE de solo +6.8% (0.249 → 0.266) -- mucho más barato que ir al 50/50 (+11.6% de MAE para
llegar a R²=+0.046). Qué punto de la mezcla usar sigue siendo una decisión de negocio (qué tanto
pesa cada métrica para el uso final), no una que este notebook deba resolver por su cuenta.

## 4. Guardar los artefactos del ensamble

In [6]:
import json, os
import joblib

os.makedirs("../models", exist_ok=True)

# receptions / receiving_yards: se guardan los 3 modelos adicionales que completan el ensamble
# recomendado (arbol x3 + Ridge) -- el XGBoost de cada uno ya vive en {target}_xgboost.json.
for target, resultados in [("receptions", resultados_recepciones), ("receiving_yards", resultados_yardas)]:
    joblib.dump(resultados["RandomForest"]["modelo"], f"../models/{target}_randomforest.joblib")
    joblib.dump(resultados["HistGradientBoosting"]["modelo"], f"../models/{target}_histgradientboosting.joblib")
    joblib.dump(resultados["Ridge"]["modelo"], f"../models/{target}_ridge.joblib")

    metadata_ensamble = {
        "target": target,
        "ensamble_recomendado": "promedio simple de XGBoost + RandomForest + HistGradientBoosting + Ridge (pesos iguales)",
        "por_que_no_solo_arboles": "correlacion 0.97-0.99 entre XGBoost/RandomForest/HistGradientBoosting -- promediarlos solos empeora el MAE frente al mejor individual; Ridge aporta la diversidad real (correlacion ~0.93-0.95 con cada arbol)",
        "mae_mejor_individual_test_2024_2025": round(float(tabla_yardas.iloc[0]["mae"] if target == "receiving_yards" else tabla_recepciones.iloc[0]["mae"]), 4),
        "mae_ensamble_4_test_2024_2025": round(float(tabla_yardas.iloc[2]["mae"] if target == "receiving_yards" else tabla_recepciones.iloc[2]["mae"]), 4),
        "fecha_de_guardado": pd.Timestamp.now(tz="UTC").isoformat(),
        "features_arbol_en_orden": features_arbol,
        "features_ridge_en_orden": features_ridge,
    }
    with open(f"../models/{target}_ensamble_metadata.json", "w") as f:
        json.dump(metadata_ensamble, f, indent=2, ensure_ascii=False)

# receiving_tds: se guarda el modelo hurdle completo (clasificador + regresor) -- el tweedie
# ya vive en receiving_tds_xgboost.json. La mezcla (w) se deja como parametro, no fija.
clasificador_hurdle, regresor_hurdle = salida_hurdle["modelo"]
joblib.dump(clasificador_hurdle, "../models/receiving_tds_hurdle_clasificador.joblib")
joblib.dump(regresor_hurdle, "../models/receiving_tds_hurdle_regresor.joblib")

metadata_hurdle = {
    "target": "receiving_tds",
    "ensamble_recomendado": "mezcla w*tweedie + (1-w)*hurdle -- w es una decision de negocio (que tanto pesa MAE vs R2), no un valor fijo",
    "correlacion_tweedie_hurdle": round(float(np.corrcoef(pred_tweedie, pred_hurdle)[0, 1]), 3),
    "barrido_mae_r2": {
        "tweedie_puro (w=1.0)": {"mae": round(float(modeling.metricas(y_test_tds, pred_tweedie)["mae"]), 4), "r2": round(float(modeling.metricas(y_test_tds, pred_tweedie)["r2"]), 4)},
        "w=0.7": {"mae": round(float(modeling.metricas(y_test_tds, np.clip(0.7*pred_tweedie+0.3*pred_hurdle,0,None))["mae"]), 4), "r2": round(float(modeling.metricas(y_test_tds, np.clip(0.7*pred_tweedie+0.3*pred_hurdle,0,None))["r2"]), 4)},
        "w=0.5": {"mae": round(float(modeling.metricas(y_test_tds, np.clip(0.5*(pred_tweedie+pred_hurdle),0,None))["mae"]), 4), "r2": round(float(modeling.metricas(y_test_tds, np.clip(0.5*(pred_tweedie+pred_hurdle),0,None))["r2"]), 4)},
        "w=0.3": {"mae": round(float(modeling.metricas(y_test_tds, np.clip(0.3*pred_tweedie+0.7*pred_hurdle,0,None))["mae"]), 4), "r2": round(float(modeling.metricas(y_test_tds, np.clip(0.3*pred_tweedie+0.7*pred_hurdle,0,None))["r2"]), 4)},
        "hurdle_puro (w=0.0)": {"mae": round(float(salida_hurdle["test"]["mae"]), 4), "r2": round(float(salida_hurdle["test"]["r2"]), 4)},
    },
    "fecha_de_guardado": pd.Timestamp.now(tz="UTC").isoformat(),
    "features_arbol_en_orden": features_arbol,
}
with open("../models/receiving_tds_hurdle_metadata.json", "w") as f:
    json.dump(metadata_hurdle, f, indent=2, ensure_ascii=False)

print("guardados:", sorted(f for f in os.listdir("../models") if "ensamble" in f or "hurdle" in f or "randomforest" in f or "ridge" in f or "histgradient" in f))


guardados: ['receiving_tds_hurdle_clasificador.joblib', 'receiving_tds_hurdle_metadata.json', 'receiving_tds_hurdle_regresor.joblib', 'receiving_yards_ensamble_metadata.json', 'receiving_yards_histgradientboosting.joblib', 'receiving_yards_randomforest.joblib', 'receiving_yards_ridge.joblib', 'receptions_ensamble_metadata.json', 'receptions_histgradientboosting.joblib', 'receptions_randomforest.joblib', 'receptions_ridge.joblib']


## Cierre

- **`receptions`/`receiving_yards`**: promediar los 3 modelos de árbol solos (XGBoost +
  RandomForest + HistGradientBoosting) **no ayuda** -- están demasiado correlacionados
  (0.97-0.99) entre sí, promediarlos diluye al mejor con 2 un poco peores. Agregar Ridge (el
  único modelo realmente diverso, correlación ~0.93-0.95) sí mejora MAE (~0.8-1.5%) y R² en
  ambos targets -- el ensamble recomendado es árboles + Ridge, no solo árboles.
- **Lección general, no solo de este target**: un ensamble ayuda por la diversidad real entre los
  modelos que lo componen, no por sumar más modelos parecidos.
- **`receiving_tds`**: mezclar Tweedie (mejor MAE) con el modelo hurdle (mejor R², ya construido
  en `14_receiving_tds_modelos_de_conteo.ipynb`) sí tiene sentido -- son mucho más diversos entre
  sí (correlación 0.62) y su mezcla da un trade-off MAE-vs-R² **ajustable con un solo parámetro**,
  no solo 2 extremos. Es una opción concreta, nueva, para la decisión pendiente sobre este
  target -- no la resuelve por sí sola, pero la hace más manejable.
- Todos los artefactos del ensamble quedan guardados en `weekly/wr/models/` junto a los modelos
  base -- ningún modelo base fue reemplazado, esto es una capa adicional sobre lo ya guardado.